In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from io import StringIO
from pathlib import Path
import re

import numpy as np
import pandas as pd
import polars as pl
import polars.lazyframe as plf
import plotly.express as px

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True, parents=True)

RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(exist_ok=True)

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)


In [ ]:
dataset_fps = list(RAW_DIR.glob('*.parquet'))

df = pl.concat([pl.scan_parquet(fp) for fp in dataset_fps[:2]])
df.count().collect(), df.head().collect()


In [ ]:
import re
import pymorphy3

def clean_and_normalize_russian_text_advanced_slow(text: str) -> list:
    # 1. Initialize the Morphological Analyzer
    morph = pymorphy3.MorphAnalyzer()
    
    # 2. Convert to lowercase
    text_lowercase = text.lower()
    
    # 3. Advanced Tokenization via Regex
    # Pattern breakdown:
    # [а-яё]+       -> Matches one or more Russian letters
    # (?:-[а-яё]+)* -> Matches an optional internal hyphen followed by more Russian letters
    #                  The '*' allows multiple hyphens (e.g., "рок-н-ролл")
    word_pattern = r'[а-яё]+(?:-[а-яё]+)*'
    words = re.findall(word_pattern, text_lowercase)
    
    cleaned_words = []
    
    # 4. Filter by Part of Speech and Normalize
    for word in words:
        parsed = morph.parse(word)[0]
        pos = parsed.tag.POS
        
        # Keep only Nouns and Verbs (including infinitives)
        if pos in ['NOUN', 'VERB', 'INFN']:
            cleaned_words.append(parsed.normal_form)
            
    return cleaned_words

# --- Comprehensive Edge-Case Testing ---
dirty_text = """
В 2026 году бизнес-план компании "Окна-Маркет" по-прежнему лежал на столе.
Кто-то сказал: "Это же рок-н-ролл!" — и начал быстро-быстро писать код.
Из-за форс-мажора пришлось всё переделывать.
"""

result = clean_and_normalize_russian_text_advanced_slow(dirty_text)
print(result)


In [ ]:
import re
import pymorphy3


morph = pymorphy3.MorphAnalyzer()
word_pattern = re.compile(r'[а-яё]+(?:-[а-яё]+)*')


def clean_and_normalize_russian_text_advanced_fast(text: str) -> list:
    # 1. Initialize the Morphological Analyzer
    
    # 2. Convert to lowercase
    text_lowercase = text.lower()
    
    # 3. Advanced Tokenization via Regex
    # Pattern breakdown:
    # [а-яё]+       -> Matches one or more Russian letters
    # (?:-[а-яё]+)* -> Matches an optional internal hyphen followed by more Russian letters
    #                  The '*' allows multiple hyphens (e.g., "рок-н-ролл")
    words = word_pattern.findall(text_lowercase)
    
    cleaned_words = []
    
    # 4. Filter by Part of Speech and Normalize
    for word in words:
        parsed = morph.parse(word)[0]
        pos = parsed.tag.POS
        
        # Keep only Nouns and Verbs (including infinitives)
        if pos in ['NOUN', 'VERB', 'INFN']:
            cleaned_words.append(parsed.normal_form)
            
    return cleaned_words

# --- Comprehensive Edge-Case Testing ---
dirty_text = """
В 2026 году бизнес-план компании "Окна-Маркет" по-прежнему лежал на столе.
Кто-то сказал: "Это же рок-н-ролл!" — и начал быстро-быстро писать код.
Из-за форс-мажора пришлось всё переделывать.
"""

result = clean_and_normalize_russian_text_advanced_fast(dirty_text)
print(result)


In [ ]:
# import re
# import unicodedata
# import spacy

# spacy.require_gpu()

# nlp = spacy.load("ru_core_news_sm", disable=["parser", "ner"])
# nlp = spacy.load("ru_core_news_md", disable=["parser", "ner"])

# # Regex pattern to match individual Cyrillic words
# RU_WORD_RE = re.compile(r'^[а-яёА-ЯЁ]+$')


# def extract_russian_nouns_and_verbs(text: str) -> list:
#     if not text:
#         return []

#     # 3. Unicode Normalization
#     # NFKD decomposes combined characters (e.g., 'а́' becomes 'а' + '\u0301')
#     decomposed = unicodedata.normalize('NFKD', text)
#     # Strip away the Combining Acute Accent (\u0301) which represents the stress mark
#     stripped = "".join([c for c in decomposed if ord(c) != 0x0301])
#     # Recompose back into standard standard format (NFC)
#     cleaned_text = unicodedata.normalize('NFC', stripped)

#     # 4. Process text through the GPU-accelerated pipeline
#     doc = nlp(cleaned_text)
    
#     tokens = list(doc)
#     results = []
#     i = 0
#     n = len(tokens)

#     # 5. Extract and reconstruct compound words
#     while i < n:
#         token = tokens[i]

#         # Check if the token is written in the Russian alphabet
#         if RU_WORD_RE.match(token.text):
#             current_group = [token]

#             # Look ahead to catch hyphenated compound words with no spaces (e.g., "диваны-кровати")
#             while (i + 2 < n and 
#                    tokens[i].whitespace_ == "" and 
#                    tokens[i+1].text == "-" and 
#                    tokens[i+1].whitespace_ == "" and 
#                    RU_WORD_RE.match(tokens[i+2].text)):
#                 current_group.extend([tokens[i+1], tokens[i+2]])
#                 i += 2  # Advance past the hyphen and the next sub-word

#             # Isolate the linguistic components (ignoring the hyphens)
#             word_tokens = [t for t in current_group if t.text != "-"]

#             # Filter for Nouns and Verbs
#             # SpaCy POS tags: NOUN (nouns), PROPN (proper nouns), VERB (verbs), AUX (auxiliary verbs)
#             is_target_pos = any(t.pos_ in {'NOUN', 'PROPN', 'VERB', 'AUX'} for t in word_tokens)

#             if is_target_pos:
#                 # Reconstruct the normal form (lemma) of the compound word
#                 lemma = "".join([t.lemma_ for t in current_group])
#                 results.append(lemma)

#             i += 1
#         else:
#             i += 1

#     return results

# extract_russian_nouns_and_verbs(dirty_text)


In [ ]:
import time
from tqdm import tqdm


DATA_SKIP = 5000
DATA_TAKE = 10

target_texts = df.slice(DATA_SKIP, DATA_TAKE).select('text').collect()['text'].to_list()

LAUNCH_COUNT = 10


def test_function(func):
    res = []
    func_name = func.__name__

    for i in tqdm(range(LAUNCH_COUNT), desc=f'Testing {func_name}'):
        start = time.perf_counter()
        
        [func(cur_text) for cur_text in target_texts]
        
        end = time.perf_counter()
        res.append(end - start)
    
    return dict(name=func_name, result=res)


results = []
results.append(test_function(clean_and_normalize_russian_text_advanced_slow))
results.append(test_function(clean_and_normalize_russian_text_advanced_fast))


In [ ]:
results

In [ ]:
px.histogram(
    results[0]['result'],
    nbins=100,
    histnorm='probability density',
    title='Execution Time Distribution',
    labels={'value': 'Time (seconds)', 'count': 'Probability density'},
).data[0]


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


# px.scatter(
#     [results[i]['result'] for i in range(2)],
#     # nbins=100,
#     # histnorm='probability density',
#     title='Execution Time Distribution',
#     labels={'value': 'Time (seconds)', 'count': 'Probability density'},
# ).show()

px.histogram(
    results[0]['result'],
    nbins=100,
    histnorm='probability density',
    title='Execution Time Distribution',
    labels={'value': 'Time (seconds)', 'count': 'Probability density'},
).show()

px.histogram(
    results[1]['result'],
    nbins=100,
    histnorm='probability density',
    title='Execution Time Distribution',
    labels={'value': 'Time (seconds)', 'count': 'Probability density'},
).show()
